In [11]:
#!/usr/bin/env python3
import numpy as np
from scipy.optimize import linear_sum_assignment

def generate_dataset(n_reviewers=30, n_papers=20, lambda_=3, mu_=5):
    """
    Generates a synthetic matrix simulating reviewer-to-paper similarity scores.
    Includes the bottleneck test case at Paper 0.
    """
    np.random.seed(98)
    S = np.random.uniform(0.05, 0.95, (n_reviewers, n_papers))

    # Construct the bottleneck test case
    S[:, 0] = np.random.uniform(0.05, 0.2, n_reviewers)
    S[0, 0] = 0.90
    S[1, 0] = 0.85

    return S

def evaluate_metrics(S, assignment):
    """
    Computes Max-Min fairness and Nash Social Welfare.
    """
    m = S.shape[1]
    valuations = [sum(S[i, j] for i in assignment[j]) for j in range(m)]

    max_min_fairness = min(valuations)
    nsw = np.prod([max(v, 1e-5) for v in valuations]) ** (1.0 / m)

    return max_min_fairness, nsw

def run_approx(S, lambda_, mu_, epsilon=0.01):
    """
    Implements the 3-Phase (6 + e)-Approximation Algorithm for EXACT CAPACITATED ONE-SIDED NSW.

    Parameters:
    -----------
    S       : Similarity matrix (Reviewers x Papers)
    lambda_ : Demand per paper (capacity of the agent in the paper's terminology)
    mu_     : Load per reviewer (handled via virtual expansion)
    epsilon : Accuracy parameter for local search convergence
    """
    n_reviewers, n_papers = S.shape
    n_virtual_reviewers = n_reviewers * mu_

    # Helper to map virtual reviewer ID back to actual reviewer ID
    def actual_id(vr_id):
        return vr_id // mu_

    # Helper to check if a paper already has this actual reviewer assigned
    def has_duplicate(paper_idx, incoming_vr, current_vrs, removing_vr=None):
        existing_actuals = {actual_id(vr) for vr in current_vrs if vr != removing_vr}
        return actual_id(incoming_vr) in existing_actuals

    # ==========================================
    # PHASE 1: Max Weight One-to-One Matching
    # ==========================================
    # We want to maximize Prod(S). Taking log turns this into a linear sum assignment.
    # Cost matrix C for SciPy (which minimizes cost, so we use negative log).
    C = np.zeros((n_papers, n_virtual_reviewers))
    for p in range(n_papers):
        for vr in range(n_virtual_reviewers):
            # Add small epsilon to prevent log(0)
            C[p, vr] = -np.log(max(S[actual_id(vr), p], 1e-5))

    row_ind, col_ind = linear_sum_assignment(C)

    # H is the set of items matched in Phase 1
    H = list(col_ind)
    # J is the set of remaining items
    J = list(set(range(n_virtual_reviewers)) - set(H))

    # ==========================================
    # PHASE 2: Local Search with Endowed Valuations
    # ==========================================
    R = {p: [] for p in range(n_papers)}

    # Compute l(p): the favorite item of paper p in J
    ell = {}
    for p in range(n_papers):
        ell[p] = max(J, key=lambda vr: S[actual_id(vr), p])

    def endowed_value(p, current_vrs):
        base_val = sum(S[actual_id(vr), p] for vr in current_vrs)
        return base_val + S[actual_id(ell[p]), p]

    # Initialize Phase 2 Allocation arbitrarily without duplicates
    unallocated_J = J.copy()
    for p in range(n_papers):
        needed = lambda_ - 1
        assigned = 0
        for vr in list(unallocated_J):
            if not has_duplicate(p, vr, R[p]):
                R[p].append(vr)
                unallocated_J.remove(vr)
                assigned += 1
            if assigned == needed:
                break

    # Local Search Loop
    while True:
        swap_made = False

        # 1. Try FULL SWAP (between two papers)
        for p1 in range(n_papers):
            for p2 in range(n_papers):
                if p1 >= p2 or swap_made: continue

                for vr1 in R[p1]:
                    for vr2 in R[p2]:
                        # Prevent duplicate reviewer assignments
                        if has_duplicate(p1, vr2, R[p1], removing_vr=vr1) or \
                           has_duplicate(p2, vr1, R[p2], removing_vr=vr2):
                            continue

                        val_p1_old = endowed_value(p1, R[p1])
                        val_p2_old = endowed_value(p2, R[p2])

                        R1_new = [x for x in R[p1] if x != vr1] + [vr2]
                        R2_new = [x for x in R[p2] if x != vr2] + [vr1]

                        val_p1_new = endowed_value(p1, R1_new)
                        val_p2_new = endowed_value(p2, R2_new)

                        # Swap condition based on geometric mean improvement
                        if (val_p1_new * val_p2_new) / (val_p1_old * val_p2_old) > (1 + epsilon):
                            R[p1] = R1_new
                            R[p2] = R2_new
                            swap_made = True
                            break
                    if swap_made: break
            if swap_made: break

        if swap_made: continue

        # 2. Try PARTIAL SWAP (between a paper and unallocated pool)
        for p in range(n_papers):
            if swap_made: break
            for vr_assigned in R[p]:
                for vr_unallocated in unallocated_J:
                    if has_duplicate(p, vr_unallocated, R[p], removing_vr=vr_assigned):
                        continue

                    val_old = endowed_value(p, R[p])
                    R_new = [x for x in R[p] if x != vr_assigned] + [vr_unallocated]
                    val_new = endowed_value(p, R_new)

                    if (val_new / val_old) > (1 + epsilon):
                        R[p] = R_new
                        unallocated_J.remove(vr_unallocated)
                        unallocated_J.append(vr_assigned)
                        swap_made = True
                        break
                if swap_made: break

        # If no FULL or PARTIAL swaps improve the objective by > 1+epsilon, we reached local optima
        if not swap_made:
            break

    # ==========================================
    # PHASE 3: Rematch Items in H
    # ==========================================
    # We now assign exactly one item from H to each paper to maximize final NSW.
    W = np.zeros((n_papers, len(H)))
    for p in range(n_papers):
        current_val = sum(S[actual_id(vr), p] for vr in R[p])
        for h_idx, h_vr in enumerate(H):
            # Infinite cost if matching this H item causes a duplicate reviewer
            if has_duplicate(p, h_vr, R[p]):
                W[p, h_idx] = 1e9
            else:
                new_val = current_val + S[actual_id(h_vr), p]
                W[p, h_idx] = -np.log(max(new_val, 1e-5))

    row_ind, col_ind = linear_sum_assignment(W)

    # Finalize Assignments
    final_assignment = {p: [] for p in range(n_papers)}
    for p in range(n_papers):
        # Add the phase 2 items
        for vr in R[p]:
            final_assignment[p].append(actual_id(vr))
        # Add the phase 3 re-matched item
        final_assignment[p].append(actual_id(H[col_ind[p]]))

    return final_assignment

# ==========================================
# EXECUTION & EXPERIMENTATION BLOCK
# ==========================================
if __name__ == "__main__":
    N_REVIEWERS = 20
    N_PAPERS = 16
    DEMAND_PER_PAPER = 3
    LOAD_PER_REVIEWER = 3

    print(f"Generating matrix for {N_REVIEWERS} Reviewers x {N_PAPERS} Papers...")
    S = generate_dataset(n_reviewers=N_REVIEWERS, n_papers=N_PAPERS,
                         lambda_=DEMAND_PER_PAPER, mu_=LOAD_PER_REVIEWER)

    print("Running 3-Phase Approximation...")
    # Using epsilon = 0.01 for a good balance of speed and optimization depth
    approx_assignment = run_approx(S, lambda_=DEMAND_PER_PAPER,
                                             mu_=LOAD_PER_REVIEWER, epsilon=0.0000001)

    approx_fairness, approx_nsw = evaluate_metrics(S, approx_assignment)

    print("\n" + "="*60)
    print("                   EXPERIMENT RESULTS")
    print("="*60)
    print(f"{'Algorithm':<25} | {'Min Fairness':<12} | {'NSW (Geom Mean)':<15}")
    print("-" * 60)
    print(f"{'Approx':<25} | {approx_fairness:<12.4f} | {approx_nsw:<15.4f}")
    print("="*60)

Generating matrix for 20 Reviewers x 16 Papers...
Running 3-Phase Approximation...

                   EXPERIMENT RESULTS
Algorithm                 | Min Fairness | NSW (Geom Mean)
------------------------------------------------------------
Approx                    | 1.9491       | 2.5212         
